# Experiment-6: GloVe (Global Vectors for Word Representation)

## 6.1. Implement the GloVe algorithm to generate word embeddings from the text corpus and find most similar words of "King".
(Implement two separate codes: a. Without using any pretrained model and b. Using Pretrained model (Gensim))

### a. Without using any pretrained model (GloVe trained from scratch on `glove_corpus.txt`)

In [1]:
import re
import numpy as np
import pandas as pd

In [2]:
with open("glove_corpus.txt", "r", encoding="utf-8") as f:
    sentences = f.read().splitlines()

def tokenize(text):
    return re.findall(r"[a-z]+", text.lower())

tokens = [tokenize(sentence) for sentence in sentences]
vocab = sorted({word for sentence in tokens for word in sentence})
word2idx = {word: i for i, word in enumerate(vocab)}
idx2word = {i: word for word, i in word2idx.items()}
V = len(vocab)

print(f"Sentences: {len(sentences)}")
print(f"Vocabulary size: {V}")
print(vocab)

Sentences: 24
Vocabulary size: 68
['a', 'and', 'around', 'at', 'books', 'brave', 'cat', 'computer', 'conducts', 'data', 'daughter', 'develops', 'doctor', 'dog', 'drinks', 'earth', 'eats', 'food', 'gives', 'heat', 'house', 'in', 'information', 'intelligent', 'is', 'kind', 'king', 'kingdom', 'laboratory', 'learn', 'light', 'live', 'lives', 'man', 'milk', 'moon', 'new', 'night', 'of', 'palace', 'patient', 'powerful', 'prince', 'princess', 'processes', 'queen', 'read', 'research', 'revolves', 'royal', 'rule', 'rules', 'school', 'scientist', 'shines', 'software', 'son', 'strong', 'students', 'sun', 'teacher', 'teaches', 'the', 'things', 'together', 'treats', 'wise', 'woman']


In [3]:
WINDOW = 5
cooccurrence = np.zeros((V, V), dtype=np.float64)

for sentence in tokens:
    indices = [word2idx[word] for word in sentence]
    for i, wi in enumerate(indices):
        for j in range(max(0, i - WINDOW), min(len(indices), i + WINDOW + 1)):
            if i == j:
                continue
            cooccurrence[wi, indices[j]] += 1.0 / abs(i - j)

print(f"Co-occurrence matrix shape: {cooccurrence.shape}")
print(f"Non-zero (word, context) pairs: {int((cooccurrence > 0).sum())}")

Co-occurrence matrix shape: (68, 68)
Non-zero (word, context) pairs: 526


In [4]:
def glove(cooccurrence, dim=50, x_max=100.0, alpha=0.75, lr0=0.05, epochs=6000, seed=42):
    rng = np.random.default_rng(seed)
    X = cooccurrence
    mask = X > 0

    log_X = np.zeros_like(X)
    log_X[mask] = np.log(X[mask])

    # Weighting function f(x) = (x / x_max)^alpha if x < x_max else 1
    weights = np.where(mask, np.where(X < x_max, (X / x_max) ** alpha, 1.0), 0.0)

    W = rng.normal(scale=0.05, size=(V, dim))   # word embeddings
    W_context = rng.normal(scale=0.05, size=(V, dim))  # context embeddings
    b_word = np.zeros(V)
    b_context = np.zeros(V)

    for epoch in range(1, epochs + 1):
        # J = sum f(X_ij) * (w_i^T w_j~ + b_i + b_j~ - log X_ij)^2
        pred = W @ W_context.T + b_word[:, None] + b_context[None, :]
        diff = np.where(mask, pred - log_X, 0.0)
        g = 2.0 * weights * diff

        lr = lr0 * max(0.0, 1.0 - epoch / epochs)
        W -= lr * (g @ W_context)
        W_context -= lr * (g.T @ W)
        b_word -= lr * g.sum(axis=1)
        b_context -= lr * g.sum(axis=0)

        if epoch % 1000 == 0 or epoch == 1:
            loss = float(np.sum(weights * diff ** 2))
            print(f"Epoch {epoch:5d} | Loss {loss:.6f}")

    embeddings = W + W_context
    return embeddings


embeddings = glove(cooccurrence)

Epoch     1 | Loss 8.800191
Epoch  1000 | Loss 0.017741
Epoch  2000 | Loss 0.001021
Epoch  3000 | Loss 0.000182


Epoch  4000 | Loss 0.000062
Epoch  5000 | Loss 0.000033
Epoch  6000 | Loss 0.000027


In [5]:
def most_similar(word, embeddings, topn=10):
    vec = embeddings[word2idx[word]]
    norms = np.linalg.norm(embeddings, axis=1) * (np.linalg.norm(vec) + 1e-9)
    sims = embeddings @ vec / norms
    order = np.argsort(-sims)

    rows = []
    for i in order:
        if idx2word[i] == word:
            continue
        rows.append({"Word": idx2word[i], "Similarity": round(float(sims[i]), 4)})
        if len(rows) == topn:
            break
    return pd.DataFrame(rows)


df_king_scratch = most_similar("king", embeddings)
df_king_scratch

,Word,Similarity
0,the,0.8189
1,lives,0.7325
2,queen,0.5854
3,is,0.5750
4,rules,0.4830
5,and,0.3541
6,kingdom,0.3497
7,princess,0.3381
8,prince,0.3296
9,sun,0.3019


### b. Using pretrained model (Gensim GloVe)

In [6]:
import gensim.downloader as api

# Pretrained GloVe vectors trained on Wikipedia + Gigaword (50-dimensional)
glove_pretrained = api.load("glove-wiki-gigaword-50")

Loaded pretrained GloVe vectors: glove-wiki-gigaword-50


In [7]:
df_king_pretrained = pd.DataFrame(
    [
        {"Word": word, "Similarity": round(float(score), 4)}
        for word, score in glove_pretrained.most_similar("king", topn=10)
    ]
)
df_king_pretrained

,Word,Similarity
0,prince,0.8236
1,queen,0.7839
2,ii,0.7746
3,emperor,0.7736
4,son,0.7667
5,uncle,0.7627
6,kingdom,0.7542
7,throne,0.7540
8,brother,0.7492
9,ruler,0.7434


### Observation
- **From scratch:** trained only on the small `glove_corpus.txt`, so neighbors of *king* reflect local co-occurrences in that corpus (e.g. *queen, kingdom, prince, princess, rules*).
- **Pretrained (Gensim):** trained on Wikipedia + Gigaword, so *king* retrieves semantically related words from large-scale statistics (e.g. *queen, prince, throne, monarch*).